In [ ]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, SBERTPairClassifier, Train, TextPairDataset
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"


/Users/oli.dmrs/Documents/Personal Projects/paraphrase-detection-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:5000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * batch_size
n_warmup_steps = steps * 0.1
n_freeze = 3

device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model = SBERTPairClassifier(model, fc_layer_sizes, device, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()

In [5]:
train_loader = DataLoader(dataset = TextPairDataset(X_train, y_train), batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = TextPairDataset(X_val, y_val), batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = TextPairDataset(X_test, y_test), batch_size = batch_size, shuffle = True, num_workers = 4)

In [ ]:
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = train_loader,
              val_dataloader = validation_loader
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()
torch.save()

2025-11-15 10:46:25.477 | INFO     | paraphrase_detection.train:run_training_loop:104 - Epoch 0: train loss = 0.49735869594618803
2025-11-15 10:46:25.478 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 0: validation loss = 0.4079992175102234
2025-11-15 10:46:25.478 | INFO     | paraphrase_detection.train:run_training_loop:106 - Epoch 0: train acc = 0.7219251336898396
2025-11-15 10:46:25.479 | INFO     | paraphrase_detection.train:run_training_loop:107 - Epoch 0: validation acc = 0.796078431372549
2025-11-15 10:46:25.479 | INFO     | paraphrase_detection.train:run_training_loop:108 - Epoch 0: validation f1 = 0.7804672108077682
2025-11-15 10:47:23.914 | INFO     | paraphrase_detection.train:run_training_loop:104 - Epoch 1: train loss = 0.3783432959746092
2025-11-15 10:47:23.916 | INFO     | paraphrase_detection.train:run_training_loop:105 - Epoch 1: validation loss = 0.39548199623823166
2025-11-15 10:47:23.917 | INFO     | paraphrase_detection.train:run_training_loo